# 27 — Sequence-to-Sequence Encoder–Decoder Mechanics

**Learning objective.** Understand encoder/decoder state flow, teacher forcing and autoregressive decoding with a tiny PyTorch sequence transduction task.

This notebook follows the track contract: concept → inspectable implementation → rendered result → failure modes → production implication.

In [1]:
from pathlib import Path
import re, random, math, json
import numpy as np
import pandas as pd
np.random.seed(42); random.seed(42)
print("Reproducibility seed: 42")

Reproducibility seed: 42


Seq2seq maps one variable-length sequence to another. Classic neural machine translation used an RNN encoder and decoder; modern translation usually uses encoder–decoder transformers. The training/inference distinction is fundamental: during training the decoder can receive the true previous target (**teacher forcing**), while inference feeds back its own prediction.

In [2]:
import torch, torch.nn as nn
torch.manual_seed(42)
# Toy transduction: reverse a length-3 digit sequence. This makes alignment inspectable.
train=[]
for a in range(1,5):
  for b in range(1,5):
    for c in range(1,5): train.append(([a,b,c],[c,b,a]))
X=torch.tensor([x for x,y in train]); Y=torch.tensor([y for x,y in train])
class Seq2Seq(nn.Module):
    def __init__(self,vocab=6,d=16,h=24):
        super().__init__(); self.emb=nn.Embedding(vocab,d); self.enc=nn.GRU(d,h,batch_first=True); self.dec=nn.GRU(d,h,batch_first=True); self.out=nn.Linear(h,vocab)
    def forward(self,x,y):
        _,h=self.enc(self.emb(x)); dec_in=torch.cat([torch.zeros((len(y),1),dtype=torch.long),y[:,:-1]],dim=1)
        z,_=self.dec(self.emb(dec_in),h); return self.out(z)
model=Seq2Seq(); opt=torch.optim.Adam(model.parameters(),lr=.03); lossfn=nn.CrossEntropyLoss()
for epoch in range(120):
    opt.zero_grad(); logits=model(X,Y); loss=lossfn(logits.reshape(-1,6),Y.reshape(-1)); loss.backward(); opt.step()
with torch.no_grad(): pred=model(X,Y).argmax(-1); acc=(pred==Y).float().mean().item()
print('final training loss:',round(loss.item(),4),'token accuracy:',round(acc,3))

final training loss: 0.0007 token accuracy: 1.0


In [3]:
# Greedy autoregressive decoding: decoder consumes its own previous prediction.
def decode(x,steps=3):
    x=torch.tensor([x]); _,h=model.enc(model.emb(x)); prev=torch.zeros((1,1),dtype=torch.long); out=[]
    for _ in range(steps):
        z,h=model.dec(model.emb(prev),h); token=model.out(z[:,-1]).argmax(-1); out.append(int(token)); prev=token.view(1,1)
    return out
for x in ([1,2,3],[4,1,2],[3,3,1]): print(x,'->',decode(x),'expected',list(reversed(x)))

[1, 2, 3] -> [3, 2, 1] expected [3, 2, 1]
[4, 1, 2] -> [2, 1, 4] expected [2, 1, 4]
[3, 3, 1] -> [1, 3, 3] expected [1, 3, 3]


Exposure bias arises because training with teacher forcing conditions on correct history while inference conditions on model-generated history. Attention and transformers largely replace a single fixed encoder state with direct access to richer source representations.

---
## Production takeaways
- Treat decoding, privacy, robustness and task heads as explicit system design choices.
- Keep evaluation aligned with the actual task and deployment risk.